# U-Net baseline trên Kvasir-SEG

Notebook này kiểm tra dữ liệu, train U-Net Scratch, train U-Net Pretrained ResNet34, đánh giá Dice/IoU và trực quan hóa kết quả.

In [ ]:
from pathlib import Path
import os
import sys
import torch

PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / 'scripts').is_dir() and (candidate / 'models').is_dir():
        PROJECT_ROOT = candidate
        break

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'Kvasir-SEG'
print('Project:', PROJECT_ROOT)
print('Dataset:', DATA_DIR)
print('PyTorch:', torch.__version__)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')
assert (DATA_DIR / 'images').is_dir()
assert (DATA_DIR / 'masks').is_dir()

In [ ]:
from utils.dataset import find_image_mask_pairs, get_dataloaders

image_paths, mask_paths = find_image_mask_pairs(str(DATA_DIR))
print('Image-mask pairs:', len(image_paths))
assert len(image_paths) > 0

train_loader, val_loader, test_loader = get_dataloaders(
    data_dir=str(DATA_DIR),
    img_size=(256, 256),
    batch_size=8,
    train_ratio=0.8,
    val_ratio=0.1,
    num_workers=0,
    seed=42,
)

In [ ]:
# Train U-Net Scratch
import subprocess

subprocess.run([
    sys.executable, 'scripts/train.py',
    '--model', 'scratch',
    '--data_dir', str(DATA_DIR),
    '--epochs', '20',
    '--batch_size', '8',
    '--img_size', '256',
    '--lr', '1e-4',
    '--save_dir', 'experiments/unet_scratch/checkpoints',
], cwd=PROJECT_ROOT, check=True)

In [ ]:
# Train U-Net Pretrained ResNet34
subprocess.run([
    sys.executable, 'scripts/train.py',
    '--model', 'pretrained',
    '--backbone', 'resnet34',
    '--data_dir', str(DATA_DIR),
    '--epochs', '20',
    '--batch_size', '8',
    '--img_size', '256',
    '--lr', '1e-4',
    '--save_dir', 'experiments/unet_resnet34/checkpoints',
], cwd=PROJECT_ROOT, check=True)

In [ ]:
# Evaluate both models
subprocess.run([
    sys.executable, 'scripts/evaluate.py',
    '--data_dir', str(DATA_DIR),
    '--scratch_ckpt', 'experiments/unet_scratch/checkpoints/best_unet_scratch.pth',
    '--pretrained_ckpt', 'experiments/unet_resnet34/checkpoints/best_unet_pretrained_resnet34.pth',
    '--out_dir', 'experiments/comparison/results',
], cwd=PROJECT_ROOT, check=True)

In [ ]:
# Visualize curves and predictions
subprocess.run([
    sys.executable, 'scripts/visualize.py',
    '--data_dir', str(DATA_DIR),
    '--scratch_ckpt', 'experiments/unet_scratch/checkpoints/best_unet_scratch.pth',
    '--pretrained_ckpt', 'experiments/unet_resnet34/checkpoints/best_unet_pretrained_resnet34.pth',
    '--num_samples', '8',
    '--out_dir', 'experiments/comparison/results',
], cwd=PROJECT_ROOT, check=True)

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv(PROJECT_ROOT / 'experiments/comparison/results/comparison_results.csv'))
display(Image(filename=str(PROJECT_ROOT / 'experiments/comparison/results/learning_curves.png')))
display(Image(filename=str(PROJECT_ROOT / 'experiments/comparison/results/segmentation_comparison.png')))